In [13]:
import pandas as pd

df = pd.read_csv("Patient_Data.csv")

df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [17]:
df = df[['PatientID', 'Department', 'Doctor', 'BillAmount']]

df.head()

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


In [18]:
df.isnull().sum()

PatientID     0
Department    0
Doctor        0
BillAmount    2
dtype: int64

In [23]:
mean_bill = df['BillAmount'].mean()

df['BillAmount'] = df['BillAmount'].fillna(mean_bill)

In [24]:
df.isnull().sum()

PatientID     0
Department    0
Doctor        0
BillAmount    0
dtype: int64

In [25]:
df[df.duplicated(subset='PatientID', keep=False)]

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
5,101,Cardiology,Dr. Smith,5000.0


In [26]:
df = df.drop_duplicates(subset='PatientID')

df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,5925.0
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,5925.0


In [27]:
df.duplicated(subset='PatientID').sum()

np.int64(0)

In [28]:
dept_bill = df.groupby('Department')['BillAmount'].sum()

dept_bill

Department
Cardiology     11200.0
Dermatology     5925.0
Neurology       5925.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

### Department wise Total Billing

From the result we can see that Cardiology has highest total bill amount.  
Orthopedics also has high value compared to others.

Neurology and Dermatology have same values because missing bill amount was filled using mean.

This gives basic idea about revenue contribution of each department.

In [29]:
billing_df = pd.read_csv("Billing_Data.csv")

billing_df.head()

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [30]:
billing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   PatientID         5 non-null      int64
 1   InsuranceCovered  5 non-null      int64
 2   FinalAmount       5 non-null      int64
dtypes: int64(3)
memory usage: 252.0 bytes


In [31]:
merged_df = pd.merge(df, billing_df, on='PatientID', how='inner')

merged_df

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.0,2000,3000
1,102,Neurology,Dr. John,5925.0,1500,3500
2,103,Orthopedics,Dr. Lee,7500.0,2500,5000
3,104,Cardiology,Dr. Smith,6200.0,3000,3200
4,105,Dermatology,Dr. Rose,5925.0,1000,4000


### Merging Datasets

Here we merged patient dataset with billing dataset using PatientID.  
We used inner join so only matching records are included.

After merging, we get combined data with billing details like insurance and final amount.

In [32]:
merged_df.shape

(5, 6)

In [33]:
new_patients = pd.DataFrame({
    'PatientID': [106, 107],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Smith', 'Dr. John'],
    'BillAmount': [6800, 7200]
})

new_patients

,PatientID,Department,Doctor,BillAmount
0,106,Cardiology,Dr. Smith,6800
1,107,Neurology,Dr. John,7200


In [34]:
updated_df = pd.concat([df, new_patients], ignore_index=True)

updated_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,5925.0
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,5925.0
5,106,Cardiology,Dr. Smith,6800.0
6,107,Neurology,Dr. John,7200.0


### Adding New Patient Records

Since no extra dataset was given, we created a small sample dataset for new patients.  
Then we concatenated it with existing dataset row wise.

This simulates adding new weekly patient data.

In [37]:
extra_cols = billing_df[['InsuranceCovered', 'FinalAmount']]

final_df = pd.concat([df.reset_index(drop=True), extra_cols], axis=1)

final_df

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.0,2000,3000
1,102,Neurology,Dr. John,5925.0,1500,3500
2,103,Orthopedics,Dr. Lee,7500.0,2500,5000
3,104,Cardiology,Dr. Smith,6200.0,3000,3200
4,105,Dermatology,Dr. Rose,5925.0,1000,4000


### Adding Billing Columns

Here we added additional billing columns like InsuranceCovered and FinalAmount.  
This was done using column wise concatenation.

Now dataset contains complete billing information.

### Final Conclusion

In this assignmnt we cleaned and analyzed hospital patient data.  
Missing values in BillAmount were handled using mean and duplicate records were removed.

After merging both datasets, we got complete billing information including insurance and final amount.  

From the analysis, Cardiology department has highest billing which shows it generates more revenue.  
Neurology and Dermatology had missing values earlier which were adjusted.

Overall, the dataset is now clean and ready for further analysis like department performance or doctor wise analysis.